# Quick start — Earth Engine via the `EarthLens` facade

This notebook is the script-form quickstart for the GEE backend. The per-category
notebooks in this directory each use `earthlens.gee.GEE(...)` directly; this one
demonstrates the **unified `EarthLens` facade** entry point — the same call
pattern that works for the CHC, S3, and ECMWF backends.

It also shows the two most common follow-on patterns as commented blocks:

- bypassing the facade with `GEE(...)` directly;
- queueing an asynchronous Drive export for an AOI too large for the synchronous
  URL path.

The notebook reads service-account credentials from `GEE_SERVICE_ACCOUNT` and
`GEE_SERVICE_KEY` environment variables. See
[Service account setup](../../reference/gee/service-account-setup.md) for the
one-time provisioning steps.

## Setup

Imports and credentials up front. `earthlens` provides the unified `EarthLens` entry
point; `earthlens.gee.Catalog` is the offline band/metadata catalog. Service-account
credentials are read from `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` (with placeholder
fallbacks so the catalog cells below still run without them).

In [ ]:
import os

from earthlens.core import EarthLens

from earthlens.gee import Catalog

SERVICE_ACCOUNT = os.environ.get(
    'GEE_SERVICE_ACCOUNT', 'my-sa@my-project.iam.gserviceaccount.com'
)
SERVICE_KEY = os.environ.get('GEE_SERVICE_KEY', '/path/to/key.json')

## Browse the catalog (no network, no auth)

The catalog ships with the package and parses ~57 k lines of curated YAML in
~3 s cold / ~1 ms warm. Use it to discover what's available and look up a
band's units before issuing a download.

In [ ]:
catalog = Catalog()
print(
    f'{len(catalog.datasets)} curated datasets; '
    f'{len(catalog.available_datasets)} in the Earth Engine STAC index'
)

chirps = catalog.get_dataset('UCSB-CHG/CHIRPS/DAILY')
print(f'CHIRPS: ee_type={chirps.ee_type}, cadence={chirps.cadence}')
print(f'  bands: {list(chirps.bands)}')
print(
    f"  precipitation units: {catalog.get_band('UCSB-CHG/CHIRPS/DAILY', 'precipitation').units}"
)

## Download via the `EarthLens` facade

Monthly CHIRPS precipitation composites over a small bbox in Egypt for
Jun-Aug 2020. `EarthLens(...)` resolves `data_source="gee"` to the GEE
backend via its `_LazyRegistry`; the remaining kwargs forward verbatim to
`GEE(...)`. Output is one GeoTIFF per month (three files total) under
`out/quickstart/`.

### Build the request

`EarthLens(...)` resolves `data_source="gee"` to the GEE backend via its
`_LazyRegistry`; the remaining kwargs forward verbatim to `GEE(...)`. Keeping the
constructor on its own statement makes the request easy to read and re-run.

In [ ]:
el = EarthLens(
    data_source='gee',
    start='2020-06-01',
    end='2020-08-31',
    cadence='monthly',
    dataset='UCSB-CHG/CHIRPS/DAILY',
    variables=['precipitation'],
    aoi=[30.0, 28.0, 34.0, 32.0],
    path='out/quickstart',
    scale=5566,
)

### Authenticate

`authenticate()` resolves the service-account credentials and initialises Earth Engine.
It is kept as its own step so the request build and the network handshake stay separate.

In [ ]:
el.authenticate(
    service_account=SERVICE_ACCOUNT,
    service_key=SERVICE_KEY,
)

### Download

`download()` writes one GeoTIFF per month (three files total) under `out/quickstart/`
and returns the list of written paths.

In [ ]:
paths = el.download(progress_bar=False)
for p in paths:
    print(p)

## Alternative — bypass the facade with `GEE(...)`

The per-category notebooks in this directory all use this form. It's a thinner
call (no registry lookup) and gives the IDE precise type hints for the
backend's kwargs.

In [ ]:
# Equivalent to the cell above:
#
# from earthlens.gee import GEE
# gee = GEE(
#     start='2020-06-01', end='2020-08-31', temporal_resolution='monthly',
#     variables={'UCSB-CHG/CHIRPS/DAILY': ['precipitation']},
#     lat_lim=[28.0, 32.0], lon_lim=[30.0, 34.0],
#     path='out/quickstart', scale=5566,
#     service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY,
# )
# gee.download(progress_bar=False)

## Alternative — asynchronous Drive export for large AOIs

The default `export_via="url"` uses `getDownloadURL` which is capped at
32768 px per axis. For larger AOIs queue an
`ee.batch.Export.image.toDrive` task instead; `download()` will block while
it runs and surface the destination string when it's done.

For non-blocking submission see `track-batch-exports.ipynb` and the
per-category notebooks' "Tracking submitted jobs" sections — pass
`wait_for_export=False` and call `wait_for_task_id(...)` later.

In [ ]:
# el = EarthLens(
#     data_source='gee',
#     start='2023-01-01', end='2023-12-31', temporal_resolution='monthly',
#     variables={'COPERNICUS/S2_SR_HARMONIZED': ['B4', 'B8']},
#     lat_lim=[51.0, 53.0], lon_lim=[4.0, 7.0],
#     scale=10, export_via='drive', drive_folder='ee_exports',
#     service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY,
# )
# locations = el.download()